# Módulo 7 — Deriva y etiquetas: los dos problemas que quedaron abiertos

Los Módulos 5 y 6 llegaron a la misma conclusión por caminos independientes: **ningún umbral fijo aprendido del pasado sobrevive**. Ninguno propuso qué hacer al respecto.

Y hay un segundo hueco. El repositorio vive en dos extremos: el Módulo 1 usa las 6,3 millones de etiquetas del dataset, los Módulos 2 a 6 no usan ninguna. El caso real no se parece a ninguno de los dos — un equipo de fraude tiene un puñado de casos confirmados y millones de transacciones sin revisar.

| Método | Qué problema ataca |
|---|---|
| **Half-Space Trees** | Un detector que se actualiza solo, sin reentrenar ni etiquetar |
| **Apilado semi-supervisado** | Los 15 detectores como features de un clasificador con pocas etiquetas |
| **Aprendizaje activo** | La capacidad de revisión produce etiquetas: ¿en qué conviene gastarla? |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.adaptive.active import compare_strategies
from src.adaptive.hs_trees import HalfSpaceTrees
from src.adaptive.run_adaptive import (
    RANKING_DETECTOR,
    plot_active_learning,
    plot_label_budget,
    plot_streaming,
    streaming_experiment,
)
from src.adaptive.stacking import label_budget_curve, select_for_review, stack_scores
from src.operations.temporal import get_temporal_data
from src.unsupervised.benchmark import run_detectors
from src.unsupervised.models import anomaly_score

In [ ]:
data = get_temporal_data()
y_test, steps = data["y_test"], data["steps_test"]

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(data["X_train"])
X_test_scaled = scaler.transform(data["X_test"])
X_early_scaled = scaler.transform(data["X_early"])

print(f"Train (normales):        {X_train_scaled.shape}")
print(f"Etiquetado temprano:     {X_early_scaled.shape}, fraude={int(data['y_early'].sum())} "
      f"({data['y_early'].mean():.4%})")
print(f"Test (periodo tardio):   {X_test_scaled.shape}, fraude={int(y_test.sum())} "
      f"({y_test.mean():.4%})")

## 1. Half-Space Trees: adaptarse cuesta una pasada lineal

Los otros quince detectores comparten un supuesto: se ajustan una vez y se usan para siempre. Half-Space Trees está diseñado al revés — mantiene un **perfil de masa** de una ventana reciente que se refresca sola.

La idea es astuta: los árboles se construyen **antes de ver un solo dato**. Cada nodo parte el espacio por la mitad en una dimensión al azar, así que la estructura no depende de la muestra; lo único que se aprende es cuántos puntos caen en cada nodo. Actualizar el modelo es recontar, no reajustar.

In [ ]:
streaming = streaming_experiment(X_test_scaled, y_test, steps, data["cutoff_step"],
                                 X_train_scaled)

print(f"promedio con ventana fija:       {streaming['roc_auc_estatico'].mean():.4f}")
print(f"promedio con ventana refrescada: {streaming['roc_auc_adaptativo'].mean():.4f}")
print(f"dias evaluados: {len(streaming)}")

In [ ]:
fig = plot_streaming(streaming, output_path=None)
plt.show()

Refrescar la ventana ayuda, pero poco: 0.586 contra 0.569. Y en términos absolutos **Half-Space Trees es un detector flojo sobre PaySim** — Gaussian Mixture llega a 0.96 en el mismo split.

Vale entender por qué antes de descartarlo. Mi primera hipótesis fue la misma que rompió al VAE en el Módulo 4: las colas pesadas. HS-Trees normaliza por el min/max de la ventana, y con valores hasta ±1900 el 99% de los datos queda aplastado en una franja mínima.

**La hipótesis es falsa.** Sobre datos sintéticos con colas igual de pesadas —87% de las filas dentro del 1% del rango— HS-Trees mantiene ROC-AUC 0.985. La causa es otra.

In [ ]:
def datos_con_ruido(n_ruido, semilla=0):
    """Senal en 2 features + n_ruido columnas irrelevantes. La senal es identica entre casos."""
    senal = np.random.default_rng(semilla)
    tr = senal.normal(size=(4000, 2))
    te = np.vstack([senal.normal(size=(400, 2)), senal.normal(loc=5.0, size=(40, 2))])
    y = np.r_[np.zeros(400), np.ones(40)]
    if n_ruido == 0:
        return tr, te, y
    ruido = np.random.default_rng(semilla + 1000)
    return (np.column_stack([tr, ruido.normal(size=(4000, n_ruido))]),
            np.column_stack([te, ruido.normal(size=(440, n_ruido))]), y)


from src.unsupervised.families import GMMDensity, LODA

print(f"{'ruido':>6} {'HS-Trees':>10} {'LODA':>8} {'GMM':>8}")
for n in (0, 10, 20, 30):
    tr, te, y = datos_con_ruido(n)
    fila = [roc_auc_score(y, anomaly_score(M.fit(tr), te))
            for M in (HalfSpaceTrees(random_state=1), LODA(random_state=1), GMMDensity())]
    print(f"{n:>6} {fila[0]:>10.4f} {fila[1]:>8.4f} {fila[2]:>8.4f}")

Ahí está. **HS-Trees y LODA construyen su estructura al azar, sin mirar los datos**, así que reparten su capacidad entre todas las dimensiones por igual. Si la señal vive en unas pocas features —en PaySim está en `errorBalanceOrig` y `errorBalanceDest`— agregar columnas irrelevantes la diluye. GMM, que estima la densidad a partir de los datos, no se mueve.

Es la misma causa detrás del PR-AUC flojo de LODA en el Módulo 6. Dos módulos, un solo mecanismo, verificado en vez de supuesto.

## 2. Apilado semi-supervisado: qué hacer con unas pocas etiquetas

La idea de XGBOD: usar los scores de los quince detectores como **features adicionales** de un clasificador supervisado. Los detectores ya destilaron la estructura de "lo normal" sin gastar una etiqueta; el clasificador solo tiene que aprender a combinarlos, que es un problema mucho más chico.

Antes de mirar los números, un detalle sobre cómo se gastan las etiquetas. Etiquetar 50 transacciones al azar sobre una prevalencia del 0,08% da **0,04 fraudes esperados**: nada con qué entrenar. Un equipo real etiqueta la cola de alertas que revisa.

In [ ]:
results = run_detectors(X_train_scaled, X_test_scaled)
scores_test = {n: out["scores"] for n, out in results.items()}
scores_early = {n: out["score_fn"](X_early_scaled) for n, out in results.items()}

X_early_stacked = stack_scores(X_early_scaled, scores_early)
X_test_stacked = stack_scores(X_test_scaled, scores_test)
print(f"features: {X_early_scaled.shape[1]} originales -> {X_early_stacked.shape[1]} apiladas")

for estrategia in ("top", "random"):
    idx = select_for_review(scores_early[RANKING_DETECTOR], 100, estrategia)
    print(f"100 etiquetas por '{estrategia}': {int(np.asarray(data['y_early'])[idx].sum())} fraudes")

In [ ]:
budget = label_budget_curve(
    X_early_scaled, X_early_stacked, data["y_early"],
    X_test_scaled, X_test_stacked, y_test,
    ranking_scores=scores_early[RANKING_DETECTOR],
)
referencia = average_precision_score(y_test, scores_test[RANKING_DETECTOR])

print(f"referencia sin etiquetas (Gaussian Mixture): PR-AUC={referencia:.4f}\n")
budget[budget.estrategia == "top"]

In [ ]:
fig = plot_label_budget(budget, output_path=None)
plt.show()

Con **50 transacciones revisadas** (10 fraudes) el modelo apilado ya alcanza 0.220, casi el 0.263 del detector no supervisado que ordenó esa misma cola. Con 5.000 llega a 0.935 — más de tres veces la referencia.

Las filas de `random` son la otra mitad de la historia: hasta 500 etiquetas **no aparece un solo fraude** y no hay clasificador que entrenar. Una advertencia sobre ese bloque: la fila de 1.000 etiquetas al azar con un único positivo marca PR-AUC 0.89, y no es un resultado — con un solo ejemplo positivo la varianza es enorme y ese número es ruido.

## 3. Aprendizaje activo, y un experimento mal diseñado

La capacidad de revisión produce etiquetas, así que la pregunta operativa cambia: ya no es solo "qué alertas reviso hoy" sino "qué alertas conviene revisar para detectar mejor mañana".

La primera versión de este experimento comparaba tres estrategias: `random`, `top_score` (explotar la cola de alertas) y `uncertainty` (explorar donde el modelo duda). El resultado parecía claro y contraintuitivo: explorar ganaba, **y además encontraba el doble de fraude**.

Era un experimento mal diseñado. `top_score` ordenaba siempre por el score no supervisado —una cola estática— mientras que `uncertainty` consultaba el modelo supervisado que iba mejorando ronda a ronda. La comparación mezclaba dos variables: explotar contra explorar, **y** ranker fijo contra ranker que aprende.

La corrección es agregar `top_model`: la misma explotación, pero ordenando por el modelo que se reentrena.

In [ ]:
activo = compare_strategies(
    X_early_stacked, data["y_early"], X_test_stacked, y_test,
    unsupervised_scores=scores_early[RANKING_DETECTOR], n_rounds=8, batch_size=50,
)

print("PR-AUC en el periodo tardio:")
print(activo.pivot(index="etiquetas", columns="estrategia", values="pr_auc").to_string(
    float_format=lambda v: f"{v:.4f}"))

In [ ]:
print("Fraudes encontrados al revisar:")
print(activo.pivot(index="etiquetas", columns="estrategia",
                   values="fraudes_encontrados").to_string())

In [ ]:
fig = plot_active_learning(activo, output_path=None)
plt.show()

Con la variable aislada, la conclusión se da vuelta:

- **El salto grande está en dejar que la cola aprenda**, no en explorar: `top_score` 0.641 → `top_model` 0.941.
- **Explorar agrega poco encima de eso**: `uncertainty` 0.959 contra `top_model` 0.941, una diferencia dentro del ruido de una sola semilla.
- Y **no hay casi tensión entre explorar y explotar**: `top_model` encuentra *más* fraude que `uncertainty` (97 contra 91) mientras aprende prácticamente lo mismo.

Sin la estrategia de control habría publicado "explorar le gana a explotar", que es falso.

## 4. Conclusiones

- **Adaptarse sin etiquetas es posible y barato**, pero el detector que lo permite paga en precisión: HS-Trees promedia 0.586 contra el 0.96 de Gaussian Mixture. Refrescar la ventana cuesta una pasada lineal y no necesita una sola etiqueta.
- **La estructura construida al azar se diluye con features irrelevantes.** Es la causa común del desempeño flojo de HS-Trees acá y de LODA en el Módulo 6, verificada con un experimento controlado en vez de asumida.
- **Cincuenta etiquetas bien gastadas valen más que mil al azar.** Con prevalencia del 0,08%, muestrear al azar no encuentra positivos y no hay nada que entrenar; revisar la cola de alertas sí.
- **Los quince detectores valen como features aunque ninguno se despliegue.** El modelo apilado triplica la referencia no supervisada con 5.000 etiquetas, y ya la iguala con 50.
- **Un experimento con dos variables confundidas produce una conclusión invertida.** La estrategia de control no era un extra: era lo que distinguía el hallazgo real del espejismo.